In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import matplotlib.pyplot as plt
plt.style.use('tableau-colorblind10')

## New Keynesian Phillips curve

$\pi_t = c + \gamma_f E_t[\pi_{t+1}] + \gamma_b \pi_{t-1} + \lambda \chi_t + v_t$

In [2]:
# Load data
df_cpi = pd.read_csv('data/CPIAUCSL.csv', index_col='observation_date', parse_dates=True)
df_unemp = pd.read_csv('data/UNRATE.csv', index_col='observation_date', parse_dates=True)

# Merge and interpolate missing values in Oct 2025
df = df_cpi.join(df_unemp, how='inner').interpolate()

# Compute annualized month-over-month inflation
df['infl'] = 1200 * np.log(df['CPIAUCSL']).diff()
df = df.dropna()

In [3]:
# Prepare variables
nlag_iv = 3

df_iv = df.assign(
    infl_lead=df['infl'].shift(-1),
    **{f'infl_lag{l}':   df['infl'].shift(l)   for l in range(1, nlag_iv + 1)},
    **{f'UNRATE_lag{l}': df['UNRATE'].shift(l) for l in range(1, nlag_iv + 1)},
).dropna()
df_iv = sm.add_constant(df_iv)

instr_cols = [f'infl_lag{l}'   for l in range(2, nlag_iv + 1)] \
           + [f'UNRATE_lag{l}' for l in range(1, nlag_iv + 1)]

# IV regression
T = len(df_iv)
nw_bw = int(np.ceil(0.75 * T**(1/3)))

result = IV2SLS(
    dependent=df_iv['infl'],
    exog=df_iv[['const', 'infl_lag1']],
    endog=df_iv[['infl_lead', 'UNRATE']],
    instruments=df_iv[instr_cols],
).fit(cov_type='kernel', bandwidth=nw_bw - 1)
print(result)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                   infl   R-squared:                      0.4385
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4366
No. Observations:                 933   F-statistic:                    1041.4
Date:                Wed, Jun 03 2026   P-value (F-stat)                0.0000
Time:                        14:01:31   Distribution:                  chi2(3)
Cov. Estimator:                kernel                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const         -0.0199     0.1733    -0.1148     0.9086     -0.3596      0.3198
infl_lag1      0.2772     0.0947     2.9264     0.00

In [4]:
# Test the restriction gamma_f = gamma_b
print(result.wald_test(formula='infl_lead = infl_lag1'))

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 3.3525
P-value: 0.0671
Distributed: chi2(1)


In [5]:
# First-stage regressions
X_fs = df_iv[['const', 'infl_lag1'] + instr_cols]
for endog in ['UNRATE', 'infl_lead']:
    res_fs = sm.OLS(df_iv[endog], X_fs).fit(
        cov_type='HAC', cov_kwds={'maxlags': nw_bw - 1}
    )
    print(f'=== First stage: {endog} ===')
    print(res_fs.summary())
    print(f'\nF-stat on instruments ({", ".join(instr_cols)}):')
    print(res_fs.f_test(' = '.join(instr_cols) + ' = 0'))
    print()

=== First stage: UNRATE ===
                            OLS Regression Results                            
Dep. Variable:                 UNRATE   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     1761.
Date:                Wed, 03 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:01:31   Log-Likelihood:                -491.57
No. Observations:                 933   AIC:                             997.1
Df Residuals:                     926   BIC:                             1031.
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.1723